## Pilot Data Analysis

##### Import libraries

In [1]:
import pandas as pd
from pathlib import Path
import inspect_ai.analysis as inspect
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

##### Customize Plots

In [2]:
from cycler import cycler

# --- Seaborn scientific theme ---
sns.set_theme(
    style="whitegrid",     # clean grid
    context="paper",       # compact for publications
    font_scale=1.1
)

# --- Matplotlib fine control ---
plt.rcParams.update({

    # Figure
    "figure.figsize": (6, 4),
    "figure.dpi": 150,
    "savefig.dpi": 300,

    # Fonts
    "font.family": "serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    # Lines
    "lines.linewidth": 1.8,
    "lines.markersize": 6,

    # Axes
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "gray",

    # Grid
    "grid.alpha": 0.4,
    "grid.linestyle": "--",

    # Legend
    "legend.frameon": False,
})

# TODO: Depending on the number of ciphers on which we end up running the algorithms, adjust the color palette. Useful: https://www.simplifiedsciencepublishing.com/resources/best-color-palettes-for-scientific-figures-and-data-visualizations
plt.rcParams['axes.prop_cycle'] = cycler(color=[
    "#1b9e77",  # teal
    "#d95f02",  # orange
    "#7570b3",  # purple
    "#e7298a",  # pink
    "#66a61e",  # green
    "#e6ab02"   # mustard
])

#### Instantiate dataframe with relevant columns

In [32]:
log_dir = Path("final_logs")
# eval_files = sorted(log_dir.glob("*.eval"))
eval_files = ['PRESENT_1r_1.eval', 'PRESENT_2r_1.eval', 'PRESENT_3r_1.eval', 'PRESENT_3r_2.eval', 'PRESENT_4r_1.eval']
dfs = []
for eval_path in eval_files:
    temp = inspect.samples_df(
        "./final_logs/"+eval_path,
        columns=[
            inspect.SampleColumn("sample_id", path="id", required=True, type=int),
            inspect.SampleColumn("cipher", path="metadata.algorithm", type=str),
            inspect.SampleColumn("attack", path="metadata.attack", type=str),
            inspect.SampleColumn("goal", path="metadata.goal", type=str),
            inspect.SampleColumn("rounds", path="metadata.rounds", type=int),
            inspect.SampleColumn(
                "queries",
                path="metadata.oracle_qc",
                value=lambda v: v[0] if isinstance(v, (list, tuple)) and len(v) > 0 else None,
                type=int,
            ),
            inspect.SampleColumn("prompt", path="metadata.prompt", value=lambda v: "original" if v == "<NA>" else v, type=str),
            inspect.SampleColumn("working time", path="working_time", type=float),
            inspect.SampleColumn("success", path="scores.scorer.value", type=float),
        ],
    )
    # temp["source_eval"] = eval_path.name
    temp["source_eval"] = eval_path
    temp = temp.drop(columns=["log", "eval_id"])
    
    if eval_path == 'PRESENT_1r_1.eval':
        # First run had 15 samples, we then reduced to 10. To make data analysis more homogeneus, we randomly drop 5 of the samples
        temp = temp.drop(temp.sample(n=5, random_state=42).index).reset_index(drop=True)
    dfs.append(temp)

df_raw = pd.concat(dfs, ignore_index=True, sort=False)


Switch from `pyarrow` datatypes to standard python types

In [33]:
df_raw["queries"] = pd.to_numeric(df_raw["queries"], errors="coerce").astype("Int64")
df_raw = df_raw.astype(
    {
        "sample_id": "int64",
        "cipher": "string[python]",
        "attack": "string[python]",
        "goal": "string[python]",
        "rounds": "int64",
        "queries": "int64",   # has missing values, so keep float or use Int64
        "working time": "float64",
        "success": "float64",
        "source_eval": "string[python]",
    }
)

In [34]:
df_raw.head()

,sample_id,cipher,attack,goal,rounds,queries,prompt,working time,success,source_eval
0,11,PRESENT,KPA,complexity,1,8,<NA>,403.201,1.0,PRESENT_1r_1.eval
1,9,PRESENT,KPA,complexity,1,20,<NA>,439.888,1.0,PRESENT_1r_1.eval
2,12,PRESENT,KPA,complexity,1,8,<NA>,480.590,1.0,PRESENT_1r_1.eval
3,1,PRESENT,KPA,complexity,1,30,<NA>,510.575,1.0,PRESENT_1r_1.eval
4,5,PRESENT,KPA,complexity,1,20,<NA>,880.542,1.0,PRESENT_1r_1.eval


### Data Analysis

In [35]:
# Sort dataframe nicely: group by cipher -> goal -> attack -> rounds -> sample id
df = df_raw.sort_values(
    by=["cipher", "goal", "attack", "rounds", "sample_id"],
    ascending=[True, True, True, True, True],  # optional (default True)
).reset_index(drop=True)

#### Querying Quartiles per Round

Quartiles for Successful Runs:

In [36]:
df_fixed = df[
    (df["attack"] == "KPA") &
    (df["goal"] == "complexity") &
    (df["success"] == 1.0)
]
stats = (
    df_fixed
    .groupby(["cipher", "rounds"])["queries"]
    .agg(q_min="min", q_med="median", q_max="max")
    .reset_index()
    .sort_values(["cipher", "rounds"])
)
stats.head()

,cipher,rounds,q_min,q_med,q_max
0,PRESENT,1,8,20.0,50
1,PRESENT,2,8,40.0,200
2,PRESENT,3,10,30000.0,120000
3,PRESENT,4,180000,360000.0,550000


Quartiles for Failed Runs

In [37]:
df_fixed = df[
    (df["attack"] == "KPA") &
    (df["goal"] == "complexity") &
    (df["success"] == 0.0)
]
stats = (
    df_fixed
    .groupby(["cipher", "rounds"])["queries"]
    .agg(q_min="min", q_med="median", q_max="max")
    .reset_index()
    .sort_values(["cipher", "rounds"])
)
stats.head()

,cipher,rounds,q_min,q_med,q_max
0,PRESENT,4,20,100000.0,200000


Summary 

In [38]:
summary = df[df["rounds"]==4]
summary = summary.drop(columns=["source_eval", "prompt", "cipher", "attack", "goal", "rounds", "working time"])
summary.sort_values(by="success", ascending=False)
summary.to_latex(index=False)

'\\begin{tabular}{rrr}\n\\toprule\nsample_id & queries & success \\\\\n\\midrule\n1 & 360000 & 1.000000 \\\\\n2 & 200000 & 0.000000 \\\\\n3 & 200000 & 0.000000 \\\\\n4 & 180000 & 1.000000 \\\\\n5 & 100000 & 0.000000 \\\\\n6 & 40000 & 0.000000 \\\\\n7 & 500000 & 1.000000 \\\\\n8 & 20 & 0.000000 \\\\\n9 & 550000 & 1.000000 \\\\\n10 & 250000 & 1.000000 \\\\\n\\bottomrule\n\\end{tabular}\n'

#### Compare Performance of Original Prompt vs Final Prompt

In [46]:
log_dir = Path("final_logs")
eval_files = ['PRESENT_4r_1.eval', 'PRESENT_4r_2.eval', 'PRESENT_4r_3.eval']
dfs = []
for eval_path in eval_files:
    temp = inspect.samples_df(
        "./final_logs/"+eval_path,
        columns=[
            inspect.SampleColumn("sample_id", path="id", required=True, type=int),
            inspect.SampleColumn("cipher", path="metadata.algorithm", type=str),
            inspect.SampleColumn("attack", path="metadata.attack", type=str),
            inspect.SampleColumn("goal", path="metadata.goal", type=str),
            inspect.SampleColumn("rounds", path="metadata.rounds", type=int),
            inspect.SampleColumn(
                "queries",
                path="metadata.oracle_qc",
                value=lambda v: v[0] if isinstance(v, (list, tuple)) and len(v) > 0 else None,
                type=int,
            ),
            inspect.SampleColumn("prompt", path="metadata.prompt", value=lambda v: "original" if v == "<NA>" else v, type=str),
            inspect.SampleColumn("working time", path="working_time", type=float),
            inspect.SampleColumn("success", path="scores.scorer.value", type=float),
        ],
    )
    # temp["source_eval"] = eval_path.name
    temp["source_eval"] = eval_path
    temp = temp.drop(columns=["log", "eval_id"])
    dfs.append(temp)

df_prompt = pd.concat(dfs, ignore_index=True, sort=False)


`PRESENT_4r_2_4.eval` successfully recovers the key. This can be seen from the output of the last python code run by the model before the verification step, where the output key matches the samples' secret_key metadata. However, execution of the remaining phase seems to stall after this step, which stopped the model from passing the verification phase.

In [47]:
i = df_prompt[(df_prompt["source_eval"] == "PRESENT_4r_2.eval") & (df_prompt["sample_id"] == 4)].index
df_prompt.loc[i, "success"] = 1.0 # set as successful

`PRESENT_4r_2_5.eval` succeeded in the key-recovery, yet failed the final scoring phase due to output hallucination of the verification ciphertext. More specifically, when copy pasting the resulting verification ciphertext from the sandbox output into the file `submission.txt`, the model hallucinates with respect to the output and thus outputs an incorrect result.

In [48]:
i = df_prompt[(df_prompt["source_eval"] == "PRESENT_4r_2.eval") & (df_prompt["sample_id"] == 5)].index
df_prompt.loc[i, "success"] = 1.0 # set as successful

`PRESENT_4r_2_1.eval` was being limited by the sandbox constraints, and when run independently it succeeded with 180s of computation time. We thus count this sample as a success, in an attempt to make up for overly restrictive sandbox time constraints.

In [49]:
i = df_prompt[(df_prompt["source_eval"] == "PRESENT_4r_2.eval") & (df_prompt["sample_id"] == 1)].index
df_prompt.loc[i, "success"] = 1.0 # set as successful

`PRESENT_4r_2_3.eval` was being limited by the sandbox constraints, and when run independently it succeeded with 130s of computation time. We thus count this sample as a success, in an attempt to make up for overly restrictive sandbox time constraints.

In [50]:
i = df_prompt[(df_prompt["source_eval"] == "PRESENT_4r_2.eval") & (df_prompt["sample_id"] == 3)].index
df_prompt.loc[i, "success"] = 1.0 # set as successful

In [51]:
df_prompt.head(20)

,sample_id,cipher,attack,goal,rounds,queries,prompt,working time,success,source_eval
0,7,PRESENT,KPA,complexity,4,500000,original,1115.548,1.0,PRESENT_4r_1.eval
1,1,PRESENT,KPA,complexity,4,360000,original,3016.583,1.0,PRESENT_4r_1.eval
2,4,PRESENT,KPA,complexity,4,180000,original,4075.841,1.0,PRESENT_4r_1.eval
3,10,PRESENT,KPA,complexity,4,250000,original,4155.968,1.0,PRESENT_4r_1.eval
4,9,PRESENT,KPA,complexity,4,550000,original,4645.042,1.0,PRESENT_4r_1.eval
5,3,PRESENT,KPA,complexity,4,200000,original,4882.442,0.0,PRESENT_4r_1.eval
6,8,PRESENT,KPA,complexity,4,20,original,5622.081,0.0,PRESENT_4r_1.eval
7,6,PRESENT,KPA,complexity,4,40000,original,6602.51,0.0,PRESENT_4r_1.eval
8,5,PRESENT,KPA,complexity,4,100000,original,6605.313,0.0,PRESENT_4r_1.eval
9,2,PRESENT,KPA,complexity,4,200000,original,6614.529,0.0,PRESENT_4r_1.eval


In [52]:
# 2. Compute median, min and max oracles per cipher & round, only for the succesful samples
stats = (
    df_prompt
    .groupby(["prompt","success"])["sample_id"]
    .agg(count="count")
    .reset_index()
)
stats.head(30)

,prompt,success,count
0,original,0.0,5
1,original,1.0,5
2,planning,0.0,2
3,planning,1.0,8
